In [54]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch import nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

In [63]:
class CodeT5ForSequenceClassification(nn.Module):
    """
    Custom CodeT5+ model for sequence classification using only the encoder
    """
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.d_model, num_labels)
        self.num_labels = num_labels
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Use mean pooling over sequence dimension
        sequence_output = outputs.last_hidden_state
        
        # Apply attention mask for mean pooling
        if attention_mask is not None:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(sequence_output.size()).float()
            sum_embeddings = torch.sum(sequence_output * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        else:
            pooled_output = sequence_output.mean(dim=1)
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {'loss': loss, 'logits': logits}

In [64]:
# Setup
from transformers import T5EncoderModel
CT5P_CKPT = "../checkpoints/ct5p_only/checkpoint-2820/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")
model = CodeT5ForSequenceClassification("Salesforce/codet5p-220m", 2)
checkpoint = torch.load(os.path.join(CT5P_CKPT, "pytorch_model.bin"), map_location=DEVICE)
model.load_state_dict(checkpoint, strict=False)
model.to(DEVICE)
model.eval()

# print("✓ Full fine-tuned CodeT5+ classification model loaded.")

# 2️⃣ Extract the encoder weights only
encoder_state_dict = model.encoder.state_dict()

# 3️⃣ Load them into a fresh encoder model
base_model = T5EncoderModel.from_pretrained("Salesforce/codet5p-220m")
base_model.load_state_dict(encoder_state_dict, strict=False)
base_model.to(DEVICE)

T5EncoderModel(
  (shared): Embedding(32100, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32100, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dropout(p=0.1, 

In [65]:
# Stylometric Features
def extract_stylometric_features(code: str) -> np.ndarray:
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    num_lines = len(lines)
    num_tokens = len(code.split())
    num_chars = len(code)
    return np.array([avg_line_length, num_lines, num_tokens, num_chars], dtype=np.float32)

In [66]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [67]:
# Dataset for Feature Extraction
class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.dropna(subset=["code", "label"])
        self.data = self.data[self.data["code"].str.strip().astype(bool)]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        for _, row in self.data.iterrows():
            tokens = self.tokenizer(row["code"], padding="max_length", truncation=True, max_length=self.max_length)
            if len(tokens["input_ids"]) > 0 and sum(tokens["attention_mask"]) > 0:
                self.samples.append((tokens, row["label"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, label = self.samples[idx]
        input_ids = torch.tensor(tokens["input_ids"])
        attention_mask = torch.tensor(tokens["attention_mask"])
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [68]:
valid_df = pd.read_csv("../csvs/val.csv", usecols=["code", "label"])
valid_dataset = CodeDataset(valid_df, tokenizer)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=True)

In [69]:
# Load data
train_df = pd.read_csv("../csvs/train.csv", usecols=["code", "label"])
train_dataset = CodeDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [70]:
# Extract Features
def extract_features(model, dataset):
    loader = DataLoader(dataset, batch_size=8)
    model.eval()
    features = []
    labels = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Features"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            stylometric = batch["stylometric"].cpu().numpy()
            label_batch = batch["label"].cpu().numpy()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Use mean pooling over sequence dimension
            sequence_output = outputs.last_hidden_state
            
            # Apply attention mask for mean pooling
            if attention_mask is not None:
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(sequence_output.size()).float()
                sum_embeddings = torch.sum(sequence_output * input_mask_expanded, 1)
                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                pooled_output = sum_embeddings / sum_mask
            else:
                pooled_output = sequence_output.mean(dim=1)
            
            # Move tensors to CPU before converting to numpy
            pooled_output = pooled_output.cpu().numpy()
            
            # Now concatenate            
            combined = np.concatenate([pooled_output, stylometric], axis=1)
            features.append(combined)
            labels.extend(label_batch)
    return np.vstack(features), np.array(labels)

In [71]:
# Build correct EmbeddingDataset, extract features, and train a simple MLP classifier

class EmbeddingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512, code_col="clean_code"):
        # accept either "clean_code" or "code"
        col = code_col if code_col in dataframe.columns else ("code" if "code" in dataframe.columns else code_col)
        self.data = dataframe.dropna(subset=[col, "label"]).reset_index(drop=True)
        self.data = self.data[self.data[col].str.strip().astype(bool)]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.code_col = col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        code = row[self.code_col]
        label = int(row["label"])
        style_feat = extract_stylometric_features(code)
        tokens = self.tokenizer(code, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "stylometric": torch.tensor(style_feat, dtype=torch.float32),
            "label": label
        }

# Instantiate embedding datasets using existing dataframes
train_emb_dataset = EmbeddingDataset(train_df, tokenizer)
valid_emb_dataset = EmbeddingDataset(valid_df, tokenizer)

# Extract features (uses base_model defined earlier)
X_train, y_train = extract_features(base_model, train_emb_dataset)
X_val, y_val = extract_features(base_model, valid_emb_dataset)

# Convert to torch datasets/loaders
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_tensor_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
val_tensor_ds = torch.utils.data.TensorDataset(X_val_t, y_val_t)

batch_size = 8
train_loader_emb = DataLoader(train_tensor_ds, batch_size=batch_size, shuffle=True)
val_loader_emb = DataLoader(val_tensor_ds, batch_size=batch_size, shuffle=False)

# Simple MLP classifier
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_labels=2, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
clf = MLPClassifier(input_dim=input_dim, hidden_dim=256, num_labels=len(np.unique(y_train)))
clf.to(device)

opt = torch.optim.AdamW(clf.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    clf.train()
    total_loss = 0.0
    for xb, yb in train_loader_emb:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = clf(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
    avg_train_loss = total_loss / len(train_loader_emb.dataset)

    # Validation
    clf.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in val_loader_emb:
            xb = xb.to(device)
            logits = clf(xb)
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            preds.extend(preds_batch.tolist())
            trues.extend(yb.numpy().tolist())

    acc = accuracy_score(trues, preds)
    prec = precision_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else precision_score(trues, preds, average="macro", zero_division=0)
    rec = recall_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else recall_score(trues, preds, average="macro", zero_division=0)
    f1 = f1_score(trues, preds, average="binary", zero_division=0) if len(np.unique(y_train))==2 else f1_score(trues, preds, average="macro", zero_division=0)

    print(f"Epoch {epoch+1}/{num_epochs} - train_loss: {avg_train_loss:.4f} val_acc: {acc:.4f} val_prec: {prec:.4f} val_rec: {rec:.4f} val_f1: {f1:.4f}")

# Print final classification report
print("\nValidation classification report:")
print(classification_report(y_val, preds, zero_division=0))

Extracting Features: 100%|██████████| 235/235 [00:17<00:00, 13.65it/s]


Epoch 1/3 - train_loss: 0.5022 val_acc: 0.8787 val_prec: 0.8853 val_rec: 0.8702 val_f1: 0.8777
Epoch 2/3 - train_loss: 0.1768 val_acc: 0.8777 val_prec: 0.8884 val_rec: 0.8638 val_f1: 0.8759
Epoch 3/3 - train_loss: 0.1707 val_acc: 0.8761 val_prec: 0.8497 val_rec: 0.9138 val_f1: 0.8806

Validation classification report:
              precision    recall  f1-score   support

           0       0.91      0.84      0.87       940
           1       0.85      0.91      0.88       940

    accuracy                           0.88      1880
   macro avg       0.88      0.88      0.88      1880
weighted avg       0.88      0.88      0.88      1880

